In [0]:
CREATE CATALOG IF NOT EXISTS retail_dev;

In [0]:
%sql
SHOW CATALOGS;

In [0]:
SELECT
    current_catalog() AS current_catalog,
    current_schema() AS current_schema;

In [0]:
SHOW SCHEMAS IN retail_dev;

In [0]:
%python
print("Spark Version:", spark.version);

In [0]:
%python
## Identify the source directory

import os
from pathlib import Path

print("Current working directory:")
print(os.getcwd())

print("\nCurrent directory contents:")
print(os.listdir())

In [0]:
%python
## Now check whether the data folder exists

from pathlib import Path

data_dir = Path.cwd() / "data"

print("Data folder exists:", data_dir.exists())

if data_dir.exists():
    print("\nAvailable files:")
    for file in sorted(data_dir.glob("*.parquet")):
        print(file.name)


In [0]:
%python
## READ THE SOURCE PARQUET FILES - This will help us understand the structure of every dataset.

from pyspark.sql import functions as F
from pathlib import Path

data_dir = Path.cwd() / "data"

files = sorted(data_dir.glob("*.parquet"))

for file in files:

    print("\n" + "=" * 60)
    print("FILE:", file.name)

    df = spark.read.parquet(str(file))

    print("Total records:", df.count())

    print("Schema:")
    df.printSchema()

    print("Sample records:")
    display(df.limit(5))


In [0]:
%python
## DATA QUALITY PROFILING - This tells us how many NULL values exist in each column.

df = spark.read.parquet(
    str(data_dir / "customer_first.parquet")
)

null_counts = df.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in df.columns
])

display(null_counts)


In [0]:
%python
## IDENTIFY DUPLICATE CUSTOMER IDS


duplicate_customers = (
    df.groupBy("customer_id")
      .count()
      .filter(F.col("count") > 1)
)

display(duplicate_customers)


In [0]:
%python

customers_1 = spark.read.parquet(
    str(data_dir / "customer_first.parquet")
)

customers_2 = spark.read.parquet(
    str(data_dir / "customers_second.parquet")
)

print("First batch:", customers_1.count())
print("Second batch:", customers_2.count())

print("First batch columns:", customers_1.columns)
print("Second batch columns:", customers_2.columns)


In [0]:
%python

new_customers = customers_2.join(
    customers_1.select("customer_id"),
    on="customer_id",
    how="left_anti"
)

print("New customer records:", new_customers.count())

display(new_customers.limit(10))
